# Cookie Cats A/B Test: Should the first gate move from level 30 to level 40?

**Business question:** Does moving the first gate in Cookie Cats from level 30 to level 40 change player retention?

**Plan (written before looking at the results):**
- **Primary metric:** 7-day retention (did the player return 7 days after install?)
- **Secondary metric:** 1-day retention
- **Guardrail metric:** total game rounds played (engagement must not collapse)
- **Hypotheses:** H0: no difference in 7-day retention between versions. H1: there is a difference.
- **Significance level:** alpha = 0.05

**Data:** 90,189 players randomly assigned to `gate_30` (control) or `gate_40` (treatment). Columns: `userid`, `version`, `sum_gamerounds`, `retention_1`, `retention_7`.

## 1. Load and inspect the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chisquare, mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_confint, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

df = pd.read_csv("cookie_cats.csv")
print(df.shape)
print(df.head())
df.info()
print("userids unique:", df["userid"].is_unique)
print(df["version"].value_counts())

## 2. Validity check: sample ratio mismatch (SRM)

If assignment was a fair 50/50 split, each group should hold about half the users. A split that is too uneven can mean users were lost from one group for a systematic reason (a bug or logging problem), which would make the groups not comparable. We test observed counts against a 50/50 expectation with a chi-square goodness-of-fit test.

In [ ]:
counts = df["version"].value_counts()
expected = [len(df) / 2, len(df) / 2]
stat, p = chisquare(counts, f_exp=expected)
print(f"chi-square = {stat:.3f}, p-value = {p:.5f}")

**Reading it:** p is about 0.009. That is below 0.05 but above the stricter 0.001 threshold many teams use for SRM alerts, and the imbalance is small (about 0.9 percentage points). I treat it as a minor limitation to flag, not a reason to stop. A very small p-value (for example 1e-8) would mean stopping and investigating the assignment pipeline before analysing anything.

## 3. Explore the outcome variables

In [ ]:
print(df.groupby("version")[["retention_1", "retention_7"]].mean())
print()
print(df["sum_gamerounds"].describe())
print()
print(df.sort_values("sum_gamerounds", ascending=False).head(5))
print()
n_zero = (df["sum_gamerounds"] == 0).sum()
print(f"Users with zero rounds: {n_zero} ({n_zero / len(df):.1%})")

**Observations:**
- Raw retention is slightly lower for `gate_40` on both metrics. Whether that gap is real is what the tests below decide.
- Game rounds are heavily right-skewed (mean far above the median), and one `gate_30` user has 49,854 rounds, about 17 times the next highest. That is almost certainly a bot, test account or logging error.

**Decisions:**
- Keep the outlier for the retention tests (a yes/no outcome is barely affected) and exclude it for the rounds comparison only.
- Keep zero-round users. They were randomised like everyone else, and dropping them after the fact could introduce bias. A sensitivity check without them is in section 8.

## 4. Test the retention metrics (two-proportion z-test)

For each metric we compare retention between versions, report the difference in percentage points with a 95% confidence interval, and the p-value. Differences are `gate_40 - gate_30`, so negative means `gate_40` is worse.

In [ ]:
results = {}
for metric in ["retention_1", "retention_7"]:
    g = df.groupby("version")[metric].agg(["sum", "count"])
    z, p = proportions_ztest(g["sum"], g["count"])
    low, high = confint_proportions_2indep(
        g.loc["gate_40", "sum"], g.loc["gate_40", "count"],
        g.loc["gate_30", "sum"], g.loc["gate_30", "count"],
    )
    diff = g.loc["gate_40", "sum"] / g.loc["gate_40", "count"] - g.loc["gate_30", "sum"] / g.loc["gate_30", "count"]
    results[metric] = (diff, low, high, p)
    print(f"{metric}: diff = {diff*100:.2f} pp, 95% CI [{low*100:.2f}, {high*100:.2f}] pp, z = {z:.3f}, p = {p:.4f}")

**Reading it:**
- **7-day retention (primary):** the confidence interval is entirely below zero and p is far below 0.05, so there is good evidence that `gate_40` lowers 7-day retention, by roughly 0.8 percentage points (about 4% relative).
- **1-day retention (secondary):** the interval touches zero and p is above 0.05, so this result is inconclusive rather than evidence of no effect.
- 7-day retention was pre-specified as primary, so no multiple-testing correction is needed for it. Even a Bonferroni threshold of 0.025 for two metrics would still be passed.

### Chart: retention by version with 95% confidence intervals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=False)
for ax, metric, title in zip(axes, ["retention_1", "retention_7"], ["1-day retention", "7-day retention"]):
    g = df.groupby("version")[metric].agg(["sum", "count"])
    rates, errs = [], []
    for v in ["gate_30", "gate_40"]:
        r = g.loc[v, "sum"] / g.loc[v, "count"]
        lo, hi = proportion_confint(g.loc[v, "sum"], g.loc[v, "count"], method="wilson")
        rates.append(r * 100)
        errs.append([(r - lo) * 100, (hi - r) * 100])
    errs = np.array(errs).T
    ax.bar(["gate_30", "gate_40"], rates, yerr=errs, capsize=6, color=["#4C72B0", "#DD8452"])
    ax.set_title(title)
    ax.set_ylabel("Retention (%)")
    ax.set_ylim(min(rates) - 3, max(rates) + 3)
plt.tight_layout()
plt.show()

The y-axis is zoomed to make the small gap visible. Error bars are 95% Wilson confidence intervals for each group's own rate.

## 5. Cross-check with a bootstrap

The bootstrap re-estimates the difference by simulation instead of a formula. We resample each group with replacement 10,000 times, compute the 7-day retention difference each time, and take the middle 95%.

In [ ]:
rng = np.random.default_rng(42)

ret_30 = df.loc[df["version"] == "gate_30", "retention_7"].to_numpy().astype(int)
ret_40 = df.loc[df["version"] == "gate_40", "retention_7"].to_numpy().astype(int)

n_boot = 10_000
boot_diffs = np.empty(n_boot)
for i in range(n_boot):
    boot_diffs[i] = rng.choice(ret_40, size=len(ret_40)).mean() - rng.choice(ret_30, size=len(ret_30)).mean()

b_low, b_high = np.percentile(boot_diffs, [2.5, 97.5])
print(f"mean diff = {boot_diffs.mean()*100:.2f} pp")
print(f"95% bootstrap CI = [{b_low*100:.2f}, {b_high*100:.2f}] pp")

plt.figure(figsize=(7, 4))
plt.hist(boot_diffs * 100, bins=60, color="#4C72B0")
plt.axvline(0, color="red", linestyle="--", label="No difference")
plt.xlabel("7-day retention difference, gate_40 - gate_30 (pp)")
plt.ylabel("Number of resamples")
plt.title("Bootstrap distribution of the 7-day retention difference")
plt.legend()
plt.show()

**Reading it:** the bootstrap interval should closely match the z-test interval, and almost none of the distribution sits above zero. Two different methods agreeing means the result does not hinge on the z-test's assumptions.

## 6. Guardrail metric: game rounds played

Rounds are heavily skewed, so a t-test on means would be dominated by a few heavy players. I use the Mann-Whitney U test (rank-based) and compare medians, after dropping the single extreme outlier.

In [ ]:
df_r = df[df["sum_gamerounds"] < 40000].copy()
print("Rows dropped:", len(df) - len(df_r), "(expected: 1)")
print(df_r.groupby("version")["sum_gamerounds"].agg(["mean", "median", "count"]))

rounds_30 = df_r.loc[df_r["version"] == "gate_30", "sum_gamerounds"]
rounds_40 = df_r.loc[df_r["version"] == "gate_40", "sum_gamerounds"]
u, p = mannwhitneyu(rounds_40, rounds_30, alternative="two-sided")
print(f"Mann-Whitney U = {u:.0f}, p = {p:.4f}")

rng = np.random.default_rng(42)
a_arr, b_arr = rounds_30.to_numpy(), rounds_40.to_numpy()
med_diffs = np.empty(5000)
for i in range(5000):
    med_diffs[i] = np.median(rng.choice(b_arr, len(b_arr))) - np.median(rng.choice(a_arr, len(a_arr)))
print("95% bootstrap CI for difference in medians:", np.percentile(med_diffs, [2.5, 97.5]))

**Reading it:** means differ by about 0.04 rounds and medians by about one round. The Mann-Whitney p-value is borderline (about 0.051), which is weak evidence of a very small shift at most. Medians of whole numbers jump in steps, so the median bootstrap interval is lumpy. The guardrail is healthy: engagement did not change in any way that matters, so the retention result stands on its own.

## 7. Power analysis

Power analysis asks whether the test was large enough to detect an effect worth caring about. Sample size needed per group (80% power, alpha = 0.05) for different minimum detectable effects (MDE) on a 19.0% baseline:

In [ ]:
baseline = 0.190
analysis = NormalIndPower()
for mde in [0.002, 0.005, 0.008, 0.010]:
    es = proportion_effectsize(baseline, baseline - mde)
    n = analysis.solve_power(effect_size=es, alpha=0.05, power=0.8, ratio=1.0)
    print(f"MDE {mde*100:.1f} pp -> {n:,.0f} users per group")

**Reading it:** with about 45,000 users per group, the test could reliably detect drops of roughly 0.8 percentage points or more, but would likely miss effects under about 0.5 pp (which need around 95,000 per group). The required sample grows with the square of how small the effect is.

*Note:* I deliberately do not report "observed power" for the effect already measured. It is just a restatement of the p-value and adds no evidence.

## 8. Sensitivity check: excluding zero-round users

Does the conclusion depend on keeping users who never played a round?

In [ ]:
active = df[df["sum_gamerounds"] > 0]
g = active.groupby("version")["retention_7"].agg(["sum", "count"])
z, p = proportions_ztest(g["sum"], g["count"])
low, high = confint_proportions_2indep(
    g.loc["gate_40", "sum"], g.loc["gate_40", "count"],
    g.loc["gate_30", "sum"], g.loc["gate_30", "count"],
)
diff = g.loc["gate_40", "sum"] / g.loc["gate_40", "count"] - g.loc["gate_30", "sum"] / g.loc["gate_30", "count"]
print(f"Excluding zero-round users: diff = {diff*100:.2f} pp, 95% CI [{low*100:.2f}, {high*100:.2f}] pp, p = {p:.4f}")

Compare this with section 4. If the direction and rough size of the effect match, the conclusion does not depend on how zero-round users are handled.

## 9. Business impact

A drop of about 0.8 percentage points in 7-day retention means roughly **820 fewer players still active at day 7 per 100,000 installs**.

In [ ]:
diff7, low7, high7, p7 = results["retention_7"]
print(f"Estimated players lost per 100,000 installs: {abs(diff7)*100_000:,.0f}")
print(f"Plausible range: {abs(high7)*100_000:,.0f} to {abs(low7)*100_000:,.0f}")

## 10. Recommendation

**Keep the first gate at level 30. Do not ship `gate_40`.**

- Moving the gate to level 40 reduced 7-day retention from about 19.0% to 18.2% (roughly 4% relative). The interval excludes zero and the bootstrap agrees.
- Total play did not change meaningfully, so there is no engagement gain to offset the loss.
- Nothing in this data shows a benefit from `gate_40`.

**Risks and limitations**
- Minor group-size imbalance (SRM p about 0.009), below the usual alarm threshold but worth flagging.
- One extreme outlier excluded from the rounds comparison only.
- No revenue data, so we cannot say whether `gate_40` earns more from in-app purchases.
- Only 7 days of follow-up.
- The test could reliably detect drops of about 0.8 pp or more, but would likely miss effects under about 0.5 pp.

**Next steps**
- Obtain revenue data and check whether `gate_40` increases purchases.
- Consider testing a gate earlier than level 30.
- Run a longer-term (30-day) retention follow-up.